# AgentMailGuard — fine-tune the Layer-1 judge with Unsloth (Colab / Kaggle, T4 16 GB)

What this notebook does, end to end (about 1.5–3 h on a free T4):
1. clone the repo branch and download the public datasets (~400 MB)
2. build the L1 corpus and the chat-format SFT set (12k examples)
3. QLoRA (4-bit) fine-tune **Qwen2.5-7B-Instruct** (or Llama-3.1-8B-Instruct) with Unsloth
4. evaluate the judge on 300 validation emails (F1)
5. export a `q4_k_m` GGUF + Ollama Modelfile to Google Drive

Runtime → **T4 GPU**. For Llama-3.1 you must accept the license on Hugging Face and paste an HF token.

In [ ]:
# @title 1. Install Unsloth and clone the repo
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl peft accelerate bitsandbytes
!git clone -b feature/mailguard-defense-stack https://github.com/leluc212/AgentMailGuard.git /content/mail_guard
%cd /content/mail_guard
!pip install -e ".[eval]"

In [ ]:
# @title 2. Download datasets and build the SFT set (~10 min)
%cd /content/mail_guard
!python -m mailguard.datasets.download --all --max-mb 400
!python -m mailguard.datasets.build_l1_corpus
!python -m training.build_sft_dataset --max-per-split 12000
!wc -l datasets/processed/sft_judge/*.jsonl

In [ ]:
# @title 3. Load the base model in 4-bit and attach LoRA
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # @param ["unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"]
HF_TOKEN = ""  # @param {type:"string"}  (only needed for gated Llama weights)
MAX_SEQ_LEN = 2048

import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, token=HF_TOKEN or None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

In [ ]:
# @title 4. Prepare the chat dataset (train on the assistant JSON only)
from datasets import load_dataset
from unsloth.chat_templates import train_on_responses_only

raw = load_dataset("json", data_files={"train": "datasets/processed/sft_judge/train.jsonl", "val": "datasets/processed/sft_judge/val.jsonl"})
def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}
train_ds = raw["train"].map(fmt, remove_columns=raw["train"].column_names)
val_ds = raw["val"].select(range(300)).map(fmt, remove_columns=raw["val"].column_names)
print(train_ds[0]["text"][:800])

In [ ]:
# @title 5. Train (1 epoch ≈ 1.5–2.5 h on T4; use MAX_STEPS for a quick smoke run)
MAX_STEPS = 0  # @param {type:"integer"}  0 = full epoch; 60 = smoke test
from trl import SFTTrainer, SFTConfig
import torch

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ_LEN, packing=False,
    args=SFTConfig(
        output_dir="outputs", per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=1, max_steps=MAX_STEPS if MAX_STEPS else -1, learning_rate=2e-4,
        warmup_ratio=0.03, lr_scheduler_type="cosine", logging_steps=10, eval_strategy="steps",
        eval_steps=200, save_steps=200, save_total_limit=2, seed=42, report_to="none",
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(), optim="adamw_8bit",
    ),
)
# mask everything except the assistant turn (template markers differ per model family)
if "Qwen" in BASE_MODEL or "qwen" in BASE_MODEL:
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
else:
    trainer = train_on_responses_only(trainer, instruction_part="<|start_header_id|>user<|end_header_id|>\n\n", response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")
stats = trainer.train()
print(stats.metrics)

In [ ]:
# @title 6. Evaluate the judge on 300 validation emails (P / R / F1)
import json, re
FastLanguageModel.for_inference(model)
rows = [json.loads(l) for l in open("datasets/processed/sft_judge/val.jsonl")][:300]
tp = fp = fn = tn = 0
for r in rows:
    prompt = tokenizer.apply_chat_template(r["messages"][:-1], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=160, do_sample=False)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\"is_injection\"\s*:\s*(true|false)", text, re.I)
    pred = bool(m and m.group(1).lower() == "true")
    gold = json.loads(r["messages"][-1]["content"])["is_injection"]
    tp += pred and gold; fp += pred and not gold; fn += (not pred) and gold; tn += (not pred) and (not gold)
p = tp / max(1, tp + fp); rc = tp / max(1, tp + fn); f1 = 2 * p * rc / max(1e-9, p + rc)
print(f"n={len(rows)} precision={p:.3f} recall={rc:.3f} f1={f1:.3f} fpr={fp / max(1, fp + tn):.3f}")

In [ ]:
# @title 7. Export adapter + GGUF (q4_k_m) to Google Drive and print the Ollama Modelfile
from google.colab import drive
drive.mount("/content/drive")
NAME = "mailguard-qwen2.5-7b-judge-v1" if "Qwen" in BASE_MODEL or "qwen" in BASE_MODEL else "mailguard-llama-3.1-8b-judge-v1"
OUT = f"/content/drive/MyDrive/AgentMailGuard/{NAME}"
model.save_pretrained(f"{OUT}/adapter"); tokenizer.save_pretrained(f"{OUT}/adapter")
model.save_pretrained_gguf(f"{OUT}/gguf", tokenizer, quantization_method="q4_k_m")
modelfile = f"FROM ./{NAME}-Q4_K_M.gguf\nPARAMETER temperature 0\nPARAMETER num_ctx 8192\n"
open(f"{OUT}/gguf/Modelfile", "w").write(modelfile)
print("Saved to", OUT)
print("On your PC:  ollama create", NAME.split('-judge')[0] + ":v1", "-f Modelfile   (inside the gguf folder)")
print("Then add the tag to configs/models.yaml and set GUARD_MODELS__JUDGE in .env")